<a href="https://colab.research.google.com/github/machancejoy-max/colab-git-demo-JOY/blob/main/Assignment13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Download Alice in Wonderland (public domain)
import requests

url = "https://www.gutenberg.org/files/11/11-0.txt"
text = requests.get(url).text

print("Dataset length:", len(text))
print(text[:500])  # preview


Dataset length: 144696
*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The


In [2]:
# Basic text cleaning
text = text.lower()

# Create a character-level vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

print("Unique characters:", vocab_size)

# Create mappings
char_to_int = {c:i for i,c in enumerate(chars)}
int_to_char = {i:c for i,c in enumerate(chars)}


Unique characters: 50


In [3]:
# Convert text to integers
encoded = [char_to_int[c] for c in text]

# Create sequences for training
seq_length = 100
X = []
y = []

for i in range(0, len(encoded) - seq_length):
    X.append(encoded[i:i+seq_length])
    y.append(encoded[i+seq_length])

import numpy as np
X = np.array(X)
y = np.array(y)

print("Training samples:", X.shape)


Training samples: (144596, 100)


In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, LayerNormalization, MultiHeadAttention, Dropout
from tensorflow.keras.models import Model

# Hyperparameters
embed_dim = 64
num_heads = 4
ff_dim = 128

# Input layer
inputs = Input(shape=(seq_length,))

# Embedding
x = Embedding(vocab_size, embed_dim)(inputs)

# Transformer block
attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(x, x)
attn_output = Dropout(0.1)(attn_output)
out1 = LayerNormalization()(x + attn_output)

ff_output = Dense(ff_dim, activation="relu")(out1)
ff_output = Dense(embed_dim)(ff_output)
ff_output = Dropout(0.1)(ff_output)
out2 = LayerNormalization()(out1 + ff_output)

# Output layer
outputs = Dense(vocab_size, activation="softmax")(out2[:, -1, :])

model = Model(inputs, outputs)
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam")

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 64)   │      3,200 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 100, 64)   │     66,368 │ embedding[0][0],  │
│ (MultiHeadAttentio… │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 100, 64)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 100, 64)   │          0 │ embedding[0][0],  │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 100, 64)   │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 100, 128)  │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 100, 64)   │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 100, 64)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 100, 64)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 100, 64)   │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 64)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50)        │      3,250 │ get_item[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 89,650 (350.20 KB)

 Trainable params: 89,650 (350.20 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
    X, y,
    batch_size=128,
    epochs=5
)


Epoch 1/5
1130/1130 ━━━━━━━━━━━━━━━━━━━━ 584s 514ms/step - loss: 2.4890
Epoch 2/5
1130/1130 ━━━━━━━━━━━━━━━━━━━━ 595s 491ms/step - loss: 2.4112
Epoch 3/5
1130/1130 ━━━━━━━━━━━━━━━━━━━━ 560s 496ms/step - loss: 2.3865
Epoch 4/5
1130/1130 ━━━━━━━━━━━━━━━━━━━━ 558s 492ms/step - loss: 2.3718
Epoch 5/5
1130/1130 ━━━━━━━━━━━━━━━━━━━━ 556s 492ms/step - loss: 2.3631


In [6]:
def generate_text(model, seed_text, char_to_int, int_to_char, seq_length=100, length=300):
    # Lowercase seed
    seed_text = seed_text.lower()

    # Pad or trim seed to exactly seq_length
    if len(seed_text) < seq_length:
        seed_text = seed_text + " " * (seq_length - len(seed_text))
    else:
        seed_text = seed_text[-seq_length:]

    # Convert seed to integers
    pattern = [char_to_int[c] for c in seed_text]

    generated = seed_text

    for _ in range(length):
        x = np.reshape(pattern, (1, seq_length))
        prediction = model.predict(x, verbose=0)
        index = np.argmax(prediction)
        result = int_to_char[index]

        generated += result

        # Slide the window
        pattern.append(index)
        pattern = pattern[1:]

    return generated


In [7]:
generated = generate_text(
    model,
    seed_text="alice was beginning to get very tired",
    char_to_int=char_to_int,
    int_to_char=int_to_char
)

print(generated)



alice was beginning to get very tired                                                                                                                                                                                                                                                                                                                                                                           


In [8]:
user_input = "the rabbit looked at alice and said"
print("Generated Story:")
print(generate_text(model, user_input, char_to_int, int_to_char, length=400))


Generated Story:
the rabbit looked at alice and said                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 
